<a href="https://colab.research.google.com/github/supunabeywickrama/my-colab-work/blob/main/Audio_to_Sentence_Timestamp_Transcription_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Clean broken numpy installation
!pip uninstall -y numpy

# Install WhisperX first, then force a stable NumPy build
!pip install -q --no-cache-dir whisperx pandas
!pip install -q --no-cache-dir --force-reinstall "numpy==1.26.4"

# Restart runtime automatically
import os
os.kill(os.getpid(), 9)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 313.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6

In [1]:
import numpy as np
import pandas as pd
import whisperx
import torch

print("NumPy:", np.__version__)
print("Torch CUDA:", torch.cuda.is_available())
print("WhisperX ready")

NumPy: 1.26.4
Torch CUDA: True
WhisperX ready


In [4]:
from google.colab import files
import os

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]
print("Uploaded file:", audio_file)

Saving WhatsApp Audio 2026-05-24 at 22.07.44.mp4 to WhatsApp Audio 2026-05-24 at 22.07.44.mp4
Uploaded file: WhatsApp Audio 2026-05-24 at 22.07.44.mp4


In [5]:
import whisperx
import torch
import pandas as pd
import re
import gc

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Higher quality: "large-v3"
# If Colab gives memory error, change to "medium" or "small"
MODEL_SIZE = "large-v3"

# Reduce batch size if GPU memory error
BATCH_SIZE = 8

compute_type = "float16" if device == "cuda" else "int8"

print("Device:", device)
print("Model:", MODEL_SIZE)

# 1. Load audio
audio = whisperx.load_audio(audio_file)

# 2. Transcribe English audio
model = whisperx.load_model(
    MODEL_SIZE,
    device,
    compute_type=compute_type,
    language="en"
)

result = model.transcribe(
    audio,
    batch_size=BATCH_SIZE,
    language="en"
)

print("Detected language:", result.get("language", "en"))

# Free memory before alignment
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# 3. Align words for accurate word timestamps
model_a, metadata = whisperx.load_align_model(
    language_code="en",
    device=device
)

aligned_result = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio,
    device,
    return_char_alignments=False
)

print("Alignment finished.")

Device: cuda
Model: large-v3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

...

2026-05-24 16:53:12 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.4. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.4. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

Detected language: en
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 346MB/s]


Alignment finished.


In [6]:
def format_time(seconds):
    """Format seconds as HH:MM:SS.mmm"""
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"


def format_srt_time(seconds):
    """Format seconds as SRT time HH:MM:SS,mmm"""
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}".replace(".", ",")


def clean_sentence_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.!?;:%])", r"\1", text)
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    return text


def extract_words_with_timestamps(aligned_result):
    words = []
    word_no = 1

    for seg in aligned_result["segments"]:
        seg_start = float(seg.get("start", 0))
        seg_end = float(seg.get("end", seg_start))
        seg_words = seg.get("words", [])

        # If word timestamps are missing, fallback to segment text
        if not seg_words:
            text_words = seg.get("text", "").strip().split()
            if not text_words:
                continue

            duration = max(seg_end - seg_start, 0.01)
            step = duration / len(text_words)

            for i, word in enumerate(text_words):
                words.append({
                    "word_no": word_no,
                    "word": word.strip(),
                    "start": seg_start + i * step,
                    "end": seg_start + (i + 1) * step,
                    "estimated_time": True
                })
                word_no += 1
            continue

        # Normal case: WhisperX gives word-level timestamps
        duration = max(seg_end - seg_start, 0.01)
        fallback_step = duration / max(len(seg_words), 1)

        for i, w in enumerate(seg_words):
            word = str(w.get("word", "")).strip()
            if not word:
                continue

            start = w.get("start", None)
            end = w.get("end", None)
            estimated = False

            # Some words like numbers/symbols may not align perfectly.
            # Estimate their time instead of losing them.
            if start is None or end is None:
                start = seg_start + i * fallback_step
                end = seg_start + (i + 1) * fallback_step
                estimated = True

            words.append({
                "word_no": word_no,
                "word": word,
                "start": float(start),
                "end": float(end),
                "estimated_time": estimated
            })
            word_no += 1

    return words


def group_words_into_sentences(words, max_pause=1.2):
    sentences = []
    current = []

    abbreviations = {
        "mr.", "mrs.", "ms.", "dr.", "prof.", "sr.", "jr.",
        "vs.", "etc.", "e.g.", "i.e.", "u.s.", "u.k."
    }

    for i, word_item in enumerate(words):
        current.append(word_item)

        word_text = word_item["word"].strip()
        word_lower = word_text.lower()

        next_word = words[i + 1] if i + 1 < len(words) else None
        pause_after = 0

        if next_word:
            pause_after = next_word["start"] - word_item["end"]

        ends_with_punctuation = bool(re.search(r'[.!?]["\')\]]*$', word_text))
        is_abbreviation = word_lower in abbreviations
        long_pause = pause_after >= max_pause

        should_end = False

        if ends_with_punctuation and not is_abbreviation:
            should_end = True

        # If no punctuation, use long pause as sentence boundary
        if long_pause and len(current) >= 3:
            should_end = True

        # Last word
        if next_word is None:
            should_end = True

        if should_end:
            sentence_text = clean_sentence_text(" ".join([w["word"] for w in current]))

            sentences.append({
                "sentence_no": len(sentences) + 1,
                "start": current[0]["start"],
                "end": current[-1]["end"],
                "start_time": format_time(current[0]["start"]),
                "end_time": format_time(current[-1]["end"]),
                "duration_sec": round(current[-1]["end"] - current[0]["start"], 3),
                "text": sentence_text
            })

            current = []

    return sentences


# Extract words
words = extract_words_with_timestamps(aligned_result)

# Create word dataframe
word_df = pd.DataFrame(words)
word_df["start_time"] = word_df["start"].apply(format_time)
word_df["end_time"] = word_df["end"].apply(format_time)

# Create sentence dataframe
sentences = group_words_into_sentences(words, max_pause=1.2)
sentence_df = pd.DataFrame(sentences)

print("Total words:", len(word_df))
print("Total sentences:", len(sentence_df))

sentence_df.head(10)

Total words: 3478
Total sentences: 343


,sentence_no,start,end,start_time,end_time,duration_sec,text
0,1,3.601,24.768,00:00:03.601,00:00:24.768,21.167,And always ask for solid evidences or references.
1,2,24.788,29.470,00:00:24.788,00:00:29.470,4.682,Can you explain the second bullet point here o...
2,3,41.802,44.644,00:00:41.802,00:00:44.644,2.842,"If you're not clear about these things, make s..."
3,4,45.925,46.786,00:00:45.925,00:00:46.786,0.861,double check again.
4,5,46.866,57.574,00:00:46.866,00:00:57.574,10.708,Ask the AI tool and learn what is co-cojacent ...
5,6,64.560,66.381,00:01:04.560,00:01:06.381,1.821,So what do you mean by tile?
6,7,67.002,69.304,00:01:07.002,00:01:09.304,2.302,Tag per tile for the anomaly model.
7,8,70.004,70.925,00:01:10.004,00:01:10.925,0.921,What do you mean by tile?
8,9,91.881,98.826,00:01:31.881,00:01:38.826,6.945,so now it says typically so i'm gonna have the...
9,10,100.808,101.508,00:01:40.808,00:01:41.508,0.700,2.3 right 2.3


In [7]:
# Save word-level CSV
word_df.to_csv("word_transcript.csv", index=False)

# Save sentence-level CSV
sentence_df.to_csv("sentence_transcript.csv", index=False)

# Save readable TXT
with open("sentence_transcript.txt", "w", encoding="utf-8") as f:
    for _, row in sentence_df.iterrows():
        f.write(f"[{row['start_time']} --> {row['end_time']}] {row['text']}\n")

# Save SRT subtitle file
with open("sentence_transcript.srt", "w", encoding="utf-8") as f:
    for i, row in sentence_df.iterrows():
        f.write(f"{int(row['sentence_no'])}\n")
        f.write(f"{format_srt_time(row['start'])} --> {format_srt_time(row['end'])}\n")
        f.write(f"{row['text']}\n\n")

print("Files created:")
print("1. word_transcript.csv")
print("2. sentence_transcript.csv")
print("3. sentence_transcript.txt")
print("4. sentence_transcript.srt")

Files created:
1. word_transcript.csv
2. sentence_transcript.csv
3. sentence_transcript.txt
4. sentence_transcript.srt


In [8]:
from google.colab import files

files.download("word_transcript.csv")
files.download("sentence_transcript.csv")
files.download("sentence_transcript.txt")
files.download("sentence_transcript.srt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>